# EP3: Optimización de Hiperparámetros y Técnicas de Ensamble
### MLY1101 Machine Learning - Evaluación Parcial N°3
**Caso B:** Predicción de precios de viviendas (Ames Housing Dataset)
**Equipo N°7:** Nicolas Carrasco, Martin Corvalan, Martin Naranjo

**Objetivo de esta evaluación:**
- IE9: Optimización de hiperparámetros del modelo (20%)
- IE10: Implementación de técnicas de ensamble (30%)
- IE11: Evaluación de la capacidad de generalización del modelo (20%)
- IE12: Selección y justificación de la solución analítica óptima (30%)

**Antecedentes:** Este notebook se construye sobre el trabajo de las EV1 y EV2. Los modelos base (Ridge, Random Forest, Gradient Boosting) ya fueron entrenados y evaluados en el notebook `Housing_EV1_EV2.ipynb`.

## Plan de trabajo:

1. **Paso 1:** Nombrar explícitamente las técnicas de ensamble existentes (Bagging y Boosting)
2. **Paso 2:** Implementar ensamble que combine modelos (VotingRegressor)
3. **Paso 3:** Reforzar justificación de hiperparámetros
4. **Paso 4:** Evaluar capacidad de generalización (brecha train/test + validación cruzada)
5. **Paso 5:** Actualizar justificación de la solución óptima

---
# ══════════════════════════════════════════
# PASO 1: Técnicas de Ensamble - Bagging y Boosting
# ══════════════════════════════════════════

### Random Forest = Ensemble por Bagging (Bootstrap Aggregating)

**Bagging:**
- Entrena múltiples árboles de decisión en paralelo
- Cada árbol usa una muestra aleatoria de los datos (bootstrap)
- Cada división usa un subconjunto aleatorio de variables
- **Predicción final:** Promedio de las predicciones de todos los árboles
- **Objetivo:** Reducir la varianza (overfitting) promediando modelos independientes

**Resultado en nuestro proyecto:**
- RMSE test: US$ 25.294
- R² test: 0.921
- Robusto frente a outliers y variables poco relevantes

### Gradient Boosting = Ensemble por Boosting

**Boosting:**
- Construye árboles de forma secuencial
- Cada árbol corrige los errores del anterior
- **Predicción final:** Suma de las predicciones de todos los árboles (cada uno corrige los residuos)
- **Objetivo:** Reducir el sesgo (bias) mejorando progresivamente el modelo

**Resultado en nuestro proyecto:**
- RMSE test: US$ 19.246
- R² test: 0.954
- Mejor desempeño predictivo que Random Forest
- Más sensible a overfitting (requiere ajuste cuidadoso de hiperparámetros)

### Comparación: Bagging vs Boosting

| Aspecto | Random Forest (Bagging) | Gradient Boosting (Boosting) |
|---------|-------------------------|-------------------------------|
| Construcción | Árboles en paralelo (independientes) | Árboles en secuencia (corrigen errores) |
| Predicción final | Promedio de árboles | Suma de correcciones |
| Objetivo principal | Reducir varianza (overfitting) | Reducir sesgo (bias) |
| Velocidad | Rápido (paralelizable) | Más lento (secuencial) |
| Sensibilidad a outliers | Robusto | Sensible |
| RMSE test (nuestro caso) | US$ 25.294 | US$ 19.246 (mejor) |
| R² test (nuestro caso) | 0.921 | 0.954 (mejor) |

---
# ══════════════════════════════════════════
# PASO 2: Implementación de Ensamble Combinado (VotingRegressor)
# ══════════════════════════════════════════

### ¿Por qué VotingRegressor?

**VotingRegressor** combina múltiples modelos de regresión:
- Usa predicciones de los modelos base (Ridge, Random Forest, Gradient Boosting)
- Predicción final = promedio ponderado de las predicciones
- Es una forma explícita de ensamble que combina modelos DISTINTOS (no solo múltiples árboles del mismo tipo)
- Responde directamente a IE10: "Implementación de técnicas de ensamble"

**Alternativa:** StackingRegressor (meta-modelo que aprende a combinar) - más complejo pero podría probarse si el tiempo lo permite.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

In [2]:
# Cargar datos limpios
df = pd.read_csv('../data/housing_clean.csv')
print(f"Dataset cargado: {df.shape}")

Dataset cargado: (2927, 81)


In [3]:
# Cargar pipeline
preprocessor = joblib.load('../models/preprocessor_pipeline.pkl')
print("Pipeline cargado.")

Pipeline cargado.


In [4]:
# Separar X e y
X = df.drop(columns=['SalePrice', 'SalePrice_log'])
y = df['SalePrice_log']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (2341, 79) | Test: (586, 79)


In [5]:
# Definir modelos con hiperparámetros de EV2
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

ridge = Ridge(alpha=30, random_state=RANDOM_STATE)
rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=2, max_features=0.33, random_state=RANDOM_STATE, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8, max_features=0.5, random_state=RANDOM_STATE)

# Crear pipelines
pipe_ridge = Pipeline([('prep', preprocessor), ('model', ridge)])
pipe_rf = Pipeline([('prep', preprocessor), ('model', rf)])
pipe_gb = Pipeline([('prep', preprocessor), ('model', gb)])

# Entrenar modelos base
pipe_ridge.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
pipe_gb.fit(X_train, y_train)
print("Modelos base entrenados.")

Modelos base entrenados.


In [6]:
# Crear VotingRegressor (Ensamble)
voting_regressor = VotingRegressor(
    estimators=[
        ('ridge', pipe_ridge),
        ('random_forest', pipe_rf),
        ('gradient_boosting', pipe_gb)
    ],
    weights=[1, 1, 1]
)
print("VotingRegressor creado.")

VotingRegressor creado.


In [7]:
# Entrenar VotingRegressor
voting_regressor.fit(X_train, y_train)
print("VotingRegressor entrenado.")

VotingRegressor entrenado.


In [8]:
# Función para calcular métricas en dólares
def metricas_en_dolares(y_log_real, y_log_pred):
    y_real = np.expm1(y_log_real)
    y_pred = np.expm1(y_log_pred)
    return {
        'R2': r2_score(y_real, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
        'MAE': mean_absolute_error(y_real, y_pred),
        'MAPE': mean_absolute_percentage_error(y_real, y_pred) * 100
    }

---
# ══════════════════════════════════════════
# PASO 3: Justificación de Hiperparámetros
# ══════════════════════════════════════════

### Ridge - Hiperparámetro: alpha

**Rango explorado:** [0.1, 0.3, 1, 3, 10, 30, 100]
**Mejor valor:** 30

**Justificación:**
- alpha controla la fuerza de la regularización L2
- Valores bajos (0.1-1): casi sin regularización, riesgo de overfitting con 314 columnas
- Valores altos (100): regularización fuerte, puede subajustar
- alpha=30 da el balance óptimo: reduce coeficientes sin perder capacidad predictiva
- El rango cubre desde casi sin regularización hasta fuerte regularización

### Random Forest - Hiperparámetros

**Hiperparámetros explorados:**
- max_depth: [None, 15, 25] → mejor: 15
- min_samples_leaf: [1, 2, 4] → mejor: 2
- max_features: [0.2, 0.33, 0.5] → mejor: 0.33
- n_estimators: 200 (fijo)

**Justificación:**
- max_depth=15: limita profundidad para evitar overfitting, pero permite suficiente complejidad
- min_samples_leaf=2: requiere al menos 2 muestras por hoja, evita divisiones muy específicas
- max_features=0.33: usa 1/3 de las variables en cada división, da diversidad a los árboles
- n_estimators=200: suficiente número de árboles para estabilidad sin costo computacional excesivo

### Gradient Boosting - Hiperparámetros

**Hiperparámetros explorados:**
- n_estimators: [300, 500] → mejor: 500
- max_depth: [3, 4] → mejor: 4
- learning_rate: 0.05 (fijo)
- subsample: 0.8 (fijo)
- max_features: 0.5 (fijo)

**Justificación:**
- n_estimators=500: más árboles corrigen mejor los errores, sin overfitting excesivo
- max_depth=4: árboles poco profundos (debilitados), reducen overfitting del boosting
- learning_rate=0.05: valor conservador, cada árbol corrige solo 5% del error residual
- subsample=0.8: usa 80% de los datos por árbol, reduce overfitting mediante bootstrap
- max_features=0.5: usa 50% de variables, da diversidad sin perder información clave

### VotingRegressor - Hiperparámetros

**Hiperparámetros:**
- weights: [1, 1, 1] (pesos iguales para Ridge, RF, GB)

**Justificación:**
- Pesos iguales dan importancia similar a los tres modelos base
- Se podría optimizar (ej. [1, 2, 1] para dar más peso a RF) pero no fue el foco de esta etapa
- Los modelos base ya están optimizados individualmente, el ensamble combina sus fortalezas

